# FIT5230 M1 — Public reproducible PIGuard evaluation

This public notebook reproduces the targeted ASCII/Unicode instruction-token experiment without access to a private Google Drive. It keeps the official checkpoint, labels, prediction rule, and full evaluation sets unchanged.


## 1. Download the public project


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/dcy1279-hash/PIGuard.git"
REPO_DIR = Path("/content/PIGuard")

# Colab storage is temporary. Removing only this exact clone makes the cell rerunnable.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    check=True,
)


In [ ]:
assert REPO_DIR.is_dir(), f"Repository was not cloned: {REPO_DIR}"
os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())


## 2. Create the isolated Python 3.10 environment

The same fixed package versions are used on CPU and GPU. Only the PyTorch download index changes automatically to match the Colab runtime.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.10"], check=True)
subprocess.run(
    ["uv", "venv", "--clear", "--python", "3.10", "/content/injecguard-env"],
    check=True,
)


In [ ]:
import shutil
import subprocess
from pathlib import Path

VENV_PYTHON = "/content/injecguard-env/bin/python"

has_gpu = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
    ).returncode == 0
)
torch_index = (
    "https://download.pytorch.org/whl/cu121"
    if has_gpu
    else "https://download.pytorch.org/whl/cpu"
)

print("Runtime selected:", "GPU (CUDA 12.1)" if has_gpu else "CPU")
print("PyTorch index:", torch_index)

subprocess.run(
    [
        "uv", "pip", "install",
        "--python", VENV_PYTHON,
        "--index-url", torch_index,
        "torch==2.4.0",
    ],
    check=True,
)
subprocess.run(
    [
        "uv", "pip", "install",
        "--python", VENV_PYTHON,
        "transformers==4.44.0",
        "tokenizers==0.19.1",
        "numpy==1.26.4",
        "sentencepiece==0.2.0",
        "protobuf==4.25.4",
        "gdown==5.2.0",
    ],
    check=True,
)

checkpoint_path = Path("checkpoints/best_model.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        VENV_PYTHON, "-m", "gdown",
        "--id", "1JpiVb_wtnbBLNEjIx1KS7PHuvmARQKTu",
        "-O", str(checkpoint_path),
    ],
    check=True,
)
print("Checkpoint downloaded:", checkpoint_path)


## 3. Verify the environment and public project files


In [ ]:
import subprocess
from pathlib import Path

required_files = [
    Path("eval_m1.py"),
    Path("PIGuard.py"),
    Path("params.py"),
    Path("util.py"),
    Path("checkpoints/best_model.pth"),
    Path("datasets/BIPIA_text.json"),
    Path("datasets/BIPIA_code.json"),
]
missing = [str(path) for path in required_files if not path.exists()]
assert not missing, f"Missing required files: {missing}"

version_check = """
import torch
import transformers
import numpy
import sentencepiece
import google.protobuf

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("protobuf:", google.protobuf.__version__)
print("evaluation device:", "cuda" if torch.cuda.is_available() else "cpu")
"""

subprocess.run([VENV_PYTHON, "-c", version_check], check=True)
subprocess.run(
    [VENV_PYTHON, "-m", "py_compile", "eval_m1.py"],
    check=True,
)
print("Environment and public project files are ready.")


## 4. M1 method — one targeted token per prompt

Every BIPIA text prompt receives two deterministic variants while the model stays fixed:

- **ASCII:** familiar leetspeak substitutions such as `write → wr173`.
- **Unicode:** visually similar Cyrillic characters such as `write → wrіtе`.

The completed reference run found 29.33% original detection, 30.67% ASCII detection, and 20.00% Unicode detection. The evaluation below regenerates these metrics and reports paired prediction flips.


In [ ]:
import subprocess

example_code = r"""
from eval_m1 import m1_rewrite, m1_target_token

examples = [
    "Write a concise answer.",
    "Please show the result.",
]
for prompt in examples:
    print("Original:", prompt)
    print("Target:", m1_target_token(prompt))
    print("ASCII:", m1_rewrite(prompt, "ascii"))
    print("Unicode:", m1_rewrite(prompt, "unicode"))
    print()
"""

subprocess.run([VENV_PYTHON, "-c", example_code], check=True)


In [ ]:
import subprocess

validation_code = r"""
import json
from eval_m1 import m1_rewrite, m1_target_token

with open("datasets/BIPIA_text.json", "r", encoding="utf-8") as file:
    data = json.load(file)

original_prompts = [
    context
    for prompts in data.values()
    for context in prompts
]
target_tokens = [m1_target_token(text) for text in original_prompts]
ascii_prompts = [m1_rewrite(text, "ascii") for text in original_prompts]
unicode_prompts = [m1_rewrite(text, "unicode") for text in original_prompts]

coverage = sum(token is not None for token in target_tokens)
ascii_changed = sum(a != b for a, b in zip(original_prompts, ascii_prompts))
unicode_changed = sum(a != b for a, b in zip(original_prompts, unicode_prompts))

print("Total prompts:", len(original_prompts))
print("Target-token coverage:", f"{coverage}/{len(original_prompts)}")
print("ASCII changed:", f"{ascii_changed}/{len(original_prompts)}")
print("Unicode changed:", f"{unicode_changed}/{len(original_prompts)}")

assert len(original_prompts) == 75
assert coverage == len(original_prompts)
assert ascii_changed == len(original_prompts)
assert unicode_changed == len(original_prompts)
"""

subprocess.run([VENV_PYTHON, "-c", validation_code], check=True)


## 5. Run the full official evaluation plus M1 variants

This runs WildGuard, original/ASCII/Unicode BIPIA text, BIPIA code, and NotInject. No sampling, smaller batch, or fast mode is used. Anonymous paired results are written to `/content/m1_targeted_results.csv`.


In [ ]:
!/content/injecguard-env/bin/python eval_m1.py --resume checkpoints/best_model.pth
